# PoreCondition-Bench

**A Reproducible Deep Learning Benchmark for Facial Pore-Condition Preprocessing**

This notebook is a thin Google Colab launcher. All preparation, caching, training, reporting, and summarization logic lives in the repository's Python package. Select a GPU runtime before continuing.

**Scope:** experimental five-class pore-condition classification; not dry/oily skin typing and not a clinical system.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/porecondition-bench.git"
PROJECT_DIR = "/content/porecondition-bench"
DRIVE_DATASET_DIR = "/content/drive/MyDrive/pore_data_set_224"
CONFIG = "configs/final_protocol.json"

assert "YOUR_USERNAME" not in REPO_URL, "Set REPO_URL to your GitHub repository first."

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import pathlib
import subprocess
import sys

project = pathlib.Path(PROJECT_DIR)
if not project.exists():
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    print(f"Using existing checkout: {project}")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)

In [ ]:
from IPython.display import Markdown, display

display(Markdown(pathlib.Path("README.md").read_text(encoding="utf-8")))

In [ ]:
dataset = pathlib.Path(DRIVE_DATASET_DIR)
expected = [dataset / str(label) for label in range(1, 6)]
missing = [str(path) for path in expected if not path.is_dir()]
assert not missing, f"Missing class directories: {missing}"

data_dir = project / "data"
data_dir.mkdir(exist_ok=True)
link = data_dir / "pore_data_set_224"
if not link.exists():
    link.symlink_to(dataset, target_is_directory=True)
print(f"Dataset linked from {dataset}")

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

In [ ]:
subprocess.run(["pore-pipeline", "--config", CONFIG, "--dry-run"], check=True)

In [ ]:
# Safe after a Colab disconnect: completed work is skipped,
# and partial training resumes from its last checkpoint.
subprocess.run(["pore-pipeline", "--config", CONFIG, "--resume"], check=True)

## Held-out test

Do not evaluate the test while choosing models or preprocessing. After the repository commit, protocol, and conclusions are frozen, run the explicit `--evaluate-test` command described in the README.